# URSim Smoke Test (RTDE Receive + URScript Control)
**Updated:** 2026-03-04

Validates a URSim-in-Docker setup on macOS using:
- **RTDE receive** (state) on **30004**
- **URScript socket** (control) on **30002**
- **Dashboard** on **29999** (optional)

Use this before integrating a learned UR3 policy.


## 0) Prereqs and expected Docker port mappings

Container should expose these ports to localhost:

- **30004**: RTDE (state / telemetry)  
- **30002**: URScript (send programs / commands)  
- **29999**: Dashboard (stop/start, status)  
- **6080**: Polyscope UI (noVNC)

Example (detached):
```bash
docker run -d --name ursim --platform linux/amd64 \
  -p 127.0.0.1:6080:6080 \
  -p 127.0.0.1:5900:5900 \
  -p 127.0.0.1:30004:30004 \
  -p 127.0.0.1:30002:30002 \
  -p 127.0.0.1:29999:29999 \
  universalrobots/ursim_e-series
```

Polyscope UI:
- http://localhost:6080/vnc.html?host=localhost&port=6080


In [1]:
import socket
import time
import math
from dataclasses import dataclass

import rtde_receive


In [2]:
@dataclass
class URSimConfig:
    host: str = "127.0.0.1"
    port_rtde: int = 30004
    port_urscript: int = 30002
    port_dashboard: int = 29999
    dt: float = 0.008  # 125 Hz typical

CFG = URSimConfig()
CFG


URSimConfig(host='127.0.0.1', port_rtde=30004, port_urscript=30002, port_dashboard=29999, dt=0.008)

In [3]:
def tcp_can_connect(host: str, port: int, timeout: float = 2.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True, None
    except OSError as e:
        return False, str(e)

def check_ports(cfg: URSimConfig):
    ports = {
        "RTDE (state)": cfg.port_rtde,
        "URScript (control)": cfg.port_urscript,
        "Dashboard": cfg.port_dashboard,
    }
    for name, port in ports.items():
        ok, err = tcp_can_connect(cfg.host, port)
        if ok:
            print(f"OK   {name:18s} {cfg.host}:{port}")
        else:
            print(f"FAIL {name:18s} {cfg.host}:{port}  {err}")

check_ports(CFG)


OK   RTDE (state)       127.0.0.1:30004
OK   URScript (control) 127.0.0.1:30002
OK   Dashboard          127.0.0.1:29999


In [4]:
def dashboard_send(cmd: str, cfg: URSimConfig = CFG, timeout: float = 2.0) -> str:
    with socket.create_connection((cfg.host, cfg.port_dashboard), timeout=timeout) as s:
        banner = s.recv(4096).decode("utf-8", errors="ignore")
        s.sendall((cmd.strip() + "\n").encode("utf-8"))
        time.sleep(0.05)
        resp = s.recv(4096).decode("utf-8", errors="ignore")
    return (banner + resp).strip()

print(dashboard_send("robotmode"))
print(dashboard_send("safetystatus"))


Connected: Universal Robots Dashboard Server
Robotmode: RUNNING
Connected: Universal Robots Dashboard Server
Safetystatus: NORMAL


In [5]:
from IPython.display import clear_output
r = rtde_receive.RTDEReceiveInterface(CFG.host)
print("Connected:", r.isConnected())

for _ in range(100):
    clear_output(wait=True)
    q = r.getActualQ()
    qd = r.getActualQd()
    tcp = r.getActualTCPPose()

    print("q  =", [round(v, 4) for v in q])
    print("qd =", [round(v, 4) for v in qd])
    print("tcp=", [round(v, 4) for v in tcp])

    time.sleep(1)

    
r.disconnect()

q  = [-1.9935, -0.9137, -1.6196, -0.808, 1.5951, -0.031]
qd = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
tcp= [-0.193, -0.1101, 0.8011, -0.415, -2.2062, 1.7951]


KeyboardInterrupt: 

In [6]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)

DT = 0.005        # 200 Hz demo read loop
N_UPDATES = 10

next_t = time.perf_counter()

try:
    for i in range(N_UPDATES):
        q = r.getActualQ()
        tcp = r.getActualTCPPose()
        print(f"step {i:02d}  q={['%+.3f'%v for v in q]}  tcp_z={tcp[2]:+.3f}")

        next_t += DT
        sleep_time = next_t - time.perf_counter()
        if sleep_time > 0:
            time.sleep(sleep_time)
finally:
    r.disconnect()


step 00  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488
step 01  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488
step 02  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488
step 03  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488
step 04  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488
step 05  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488
step 06  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488
step 07  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488
step 08  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488
step 09  q=['+0.000', '-1.571', '+1.571', '+0.000', '+1.571', '+0.000']  tcp_z=+0.488


In [7]:
def send_urscript(script: str, cfg: URSimConfig = CFG, timeout: float = 2.0):
    with socket.create_connection((cfg.host, cfg.port_urscript), timeout=timeout) as s:
        s.sendall(script.encode("utf-8"))

def urscript_textmsg(msg: str) -> str:
    safe = msg.replace('"', "'")
    return f'textmsg("{safe}")'

def urscript_movej(q, a=0.5, v=0.5) -> str:
    return f"movej({q}, a={a}, v={v})"

def urscript_servoj(q, t=0.008, lookahead=0.1, gain=300) -> str:
    return f"servoj({q}, t={t}, lookahead_time={lookahead}, gain={gain})"

def urscript_program(lines, name="py_prog") -> str:
    body = "\n  ".join(lines)
    return f"""def {name}():
  {body}
end
{name}()\n"""


In [8]:
# Requires URSim in RUNNING + NORMAL state (power on, brakes released)
Q_HOME = [0.0, -math.pi/2, math.pi/2, 0.0, math.pi/2, 0.0]

prog = urscript_program([
    urscript_textmsg("hello from python (urscript smoketest)"),
    urscript_movej(Q_HOME, a=0.5, v=0.5),
], name="t")

send_urscript(prog)
print("URScript sent to 30002")


URScript sent to 30002


In [9]:
# Minimal closed-loop skeleton (functional demo):
# - read q via RTDE
# - dummy policy: hold q
# - send one servoj step as a short program
#
# NOTE: Opening a new TCP socket each cycle is NOT suitable for high-rate control.
# Keep N small and dt modest.

r = rtde_receive.RTDEReceiveInterface(CFG.host)

N = 10
dt = 0.05
try:
    for i in range(N):
        q = r.getActualQ()
        q_target = q  # placeholder
        prog = urscript_program([urscript_servoj(q_target, t=0.008, lookahead=0.1, gain=200)], name=f"step_{i}")
        send_urscript(prog)
        print(f"sent step {i:02d}")
        time.sleep(dt)
finally:
    r.disconnect()


sent step 00
sent step 01
sent step 02
sent step 03
sent step 04
sent step 05
sent step 06
sent step 07
sent step 08
sent step 09


In [10]:
# Stop helpers (Dashboard)
print(dashboard_send("stop"))


Connected: Universal Robots Dashboard Server
Stopped


## Next steps (UR3 policy integration)

For stable higher-rate control, avoid reconnecting a TCP socket each step.
Preferred patterns:
- Persistent robot-side script consuming setpoints (socket/RTDE inputs)
- Or a compatible high-rate control interface

This notebook validates:
- Port reachability
- RTDE receive works
- URScript programs execute
